# 07 · Multi-document matters — grouping work

Several documents through one `matter_id`, and the rollup it buys.

## Setup — the lab bench

In [1]:
import json
import sys
from pathlib import Path

# Work from the repo root no matter where the kernel was started.
ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
assert (ROOT / "notebooks" / "pipeline_lab.py").exists(), (
    f"llm-mailroom repo root not found above {Path.cwd()}"
)
sys.path.insert(0, str(ROOT / "notebooks"))

import pipeline_lab as lab


## What you'll see

- a 3-document matter (contract + correspondence + court opinion)
- per-doc results + the catalog rollup
- why `session_id = matter_id` matters for tracing

**Honesty label:** sequential per-file processing, exactly like the watcher's loop within a matter.

## Run the matter

In [2]:
docs = [
    (lab.DOC_CONTRACT, "msa.txt",
     lab.CLASSIFY_CONTRACT_HIGH, lab.EXTRACT_HIGH),
    (lab.DOC_CORRESPONDENCE, "grady_letter.txt",
     lab.CLASSIFY_CORRESPONDENCE_HIGH, lab.CORRESPONDENCE_EXTRACTION),
    (lab.DOC_COURT_OPINION, "order.txt",
     lab.CLASSIFY_COURT_HIGH, lab.COURT_OPINION_EXTRACTION),
]

with lab.lab_sandbox() as env:
    out = lab.run_matter(env, docs, matter_id="LAB-MATTER-100")
    for row in out["catalog_rows"]:
        print(row)
    print("rollup:", out["rollup"])


('2bcb81d8-1a5d-4305-b103-4b3ea96da15a', 'archived', 'contract', 'msa.txt')
('c00808b7-294b-4967-ba4d-ef1fc76f84d3', 'archived', 'correspondence', 'grady_letter.txt')
('508bf2f6-2489-429f-8266-16429150c637', 'archived', 'court_opinion', 'order.txt')
rollup: {'archived': 3}


## Per-document scripting

`run_matter` accepts per-doc `(text, filename, classification, extraction)` tuples — mixed-class matters serve each class its own canned extraction, the same way the legacy specialists key off the document class.

## The trace contract

Every document of the matter publishes `session_id = matter_id`, so one Langfuse session view shows the whole matter:

In [3]:
contract = lab.trace_contract(filename="msa.txt", matter_id="LAB-MATTER-100")
print("session_id:", contract["session_id"])
print("trace name:", contract["name"])
print("note:      ", contract["note"])


2026-08-24 18:17:19 [info     ] checkpointer_initialized       backend=memory


session_id: LAB-MATTER-100
trace name: document-pipeline
note:       Shape computed from the pipeline's own seeding functions — not a live Langfuse fetch.


## Where to go next

- **08 · observability_traces** — the full trace tree
- **06 · outputs_and_audit** — the per-run artifacts just written